# Визуализация данных: matplotlib и seaborn

Этот урок — самостоятельный учебник о том, как превращать таблицу в понятный ответ. Мы будем работать с локальной копией Titanic: никаких загрузок из интернета во время занятия не нужно.

**После урока вы сможете:** выбрать график под вопрос, подписать его так, чтобы он не обманывал, строить гистограммы, столбики, scatter, boxplot, линии, тепловые карты и небольшие дашборды; пользоваться и matplotlib, и seaborn; проверять себя автоматическими тестами.

## Как проходить урок

Запускайте ячейки сверху вниз (`Shift + Enter`). Перед каждым графиком сначала прочитайте **вопрос**, а уже потом код: одна и та же таблица может дать совершенно разные честные графики.

Если seaborn ещё не установлен, выполните в отдельной ячейке `pip install -r requirements.txt`. В рабочей папке уже есть `requirements.txt`, а данные лежат в `titanic.csv`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style='whitegrid', palette='deep')

# Данные рядом с ноутбуком: 891 пассажир, 15 столбцов.
df = pd.read_csv('titanic.csv')
print(df.shape)
df.head()

# 1. До графика: вопрос и данные

График — не украшение отчёта, а способ ответить на конкретный вопрос. Вот четыре разных вопроса к Titanic:

1. **Как распределён возраст?** Нужна гистограмма.
2. **Как отличается доля выживших между классами?** Нужны столбики со средними.
3. **Есть ли связь возраста и цены билета?** Нужны точки на плоскости.
4. **Как меняется доля выживших по возрастным группам?** Нужны агрегирование и линия.

Прежде чем рисовать, выясните, что хранит каждый столбец. `age` и `fare` — числовые; `sex`, `class`, `embarked` — категории; `survived` — ответ из нулей и единиц. Среднее колонки из нулей и единиц — это доля единиц. Поэтому среднее `survived` читается как доля выживших.

In [ ]:
print(df.info())
print()
print('Пропуски:')
print(df.isna().sum().sort_values(ascending=False).head())
print()
print('Категории пола:')
print(df['sex'].value_counts())
print()
print('Доля выживших по полу:')
print(df.groupby('sex')['survived'].mean())

## Мини-словарь визуализации

- **Наблюдение** — одна строка таблицы, здесь один пассажир.
- **Распределение** — какие значения встречаются часто, а какие редко.
- **Агрегация** — превращение многих строк в одно число для группы: например, средняя доля выживших в классе.
- **Выброс** — редкое необычно большое/маленькое значение. В `fare` есть очень дорогие билеты, поэтому они растягивают шкалу.
- **Кодирование цветом (`hue`)** — цвет показывает третью переменную. Он полезен, если цвет отвечает на вопрос, а не просто украшает.
- **Корреляция/связь** — два признака меняются вместе. Это не доказательство, что один признак вызывает другой.

# 2. matplotlib: рисуем осознанно

Matplotlib — базовая библиотека графиков в Python. В ней важно различать два объекта:

- `fig` (**figure**) — весь лист/картинка;
- `ax` (**axes**) — одно место с осями внутри листа.

Самый полезный шаблон:

```python
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, y)                       # рисуем данные
ax.set(title='...', xlabel='...', ylabel='...')  # объясняем читателю
plt.show()
```

Возвращать `ax` из функции удобно: в нём можно менять заголовок, пределы оси, легенду и проверять график тестом.

In [ ]:
# Анатомия одного графика: точки, подписи, сетка, легенда и аннотация.
x = np.arange(1, 6)
y = np.array([2, 3, 5, 4, 7])
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, y, marker='o', linewidth=2, color='crimson', label='наблюдения')
ax.annotate('самая высокая точка', xy=(5, 7), xytext=(3.3, 6.4), arrowprops={'arrowstyle': '->'})
ax.set(title='Пример: как устроен линейный график', xlabel='Номер дня', ylabel='Значение')
ax.set_xticks(x)
ax.legend()
ax.grid(alpha=.3)
plt.show()

## 2.1. Гистограмма: распределение одного числа

Гистограмма отвечает на вопрос «какие значения встречаются чаще?». Она делит числовую ось на интервалы — **корзинки** (`bins`) — и считает объекты в каждом.

Количество корзинок меняет впечатление: слишком мало скроет форму распределения, слишком много покажет случайный шум. Хорошая привычка — попробовать 10, 20 и 30 корзинок и убедиться, что главный вывод не меняется.

Перед гистограммой удаляем пропуски: `df['age'].dropna()`. Пропуск — не возраст 0.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, bins in zip(axes, [8, 20, 40]):
    ax.hist(df['age'].dropna(), bins=bins, color='steelblue', edgecolor='white')
    ax.set(title=f'bins = {bins}', xlabel='Возраст, лет')
axes[0].set_ylabel('Число пассажиров')
fig.suptitle('Один набор данных, три числа корзинок', y=1.03)
plt.tight_layout(); plt.show()

## 2.2. Столбики: количество и доля — не одно и то же

У столбчатой диаграммы высота может означать разные вещи.

- `value_counts()` → сколько пассажиров в каждой группе;
- `groupby(...).mean()` для `survived` → доля выживших в группе.

Это разные вопросы, поэтому у них должны быть разные подписи Y. Для доли удобно установить `ax.set_ylim(0, 1)`: 0 означает 0%, 1 — 100%.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df['class'].value_counts().reindex(['First', 'Second', 'Third'])
axes[0].bar(counts.index, counts.values, color='slateblue')
axes[0].set(title='Сколько пассажиров в каждом классе', xlabel='Класс билета', ylabel='Число пассажиров')

rates = df.groupby('class', observed=True)['survived'].mean().reindex(['First', 'Second', 'Third'])
axes[1].bar(rates.index, rates.values, color='coral')
axes[1].set(title='Доля выживших в каждом классе', xlabel='Класс билета', ylabel='Доля', ylim=(0, 1))
plt.tight_layout(); plt.show()

## 2.3. Scatter: связь двух чисел

В `scatter` каждая точка — одна строка таблицы. Ось X — один числовой признак, Y — другой. Если точек много, используйте `alpha` меньше 1: наложения станут темнее, и будет видно плотные места.

Цветом можно показать третью переменную. Здесь красный — не выжил, зелёный — выжил. Цвет позволяет увидеть группы, но не доказывает причину и не делает прогноз сам по себе.

In [ ]:
colors = df['survived'].map({0: 'tomato', 1: 'seagreen'})
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['age'], df['fare'], c=colors, alpha=.35, s=30)
ax.set(title='Возраст и цена билета; цвет — исход', xlabel='Возраст, лет', ylabel='Цена билета')
ax.scatter([], [], c='tomato', label='не выжил')
ax.scatter([], [], c='seagreen', label='выжил')
ax.legend(); plt.show()

## 2.4. Boxplot: медиана, квартили и выбросы

`boxplot` компактно сравнивает распределения. Линия в коробке — **медиана**; нижняя и верхняя границы коробки — 25-й и 75-й процентили; «усы» показывают обычный диапазон; отдельные точки часто являются выбросами.

Boxplot особенно полезен, когда нужно сравнить много групп и не хочется рисовать отдельную гистограмму для каждой. Цена билета имеет длинный хвост, поэтому также полезно посмотреть вариант с `ax.set_yscale('log')`: логарифмическая шкала сжимает огромные значения.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fare_by_class = [df.loc[df['pclass'] == c, 'fare'].dropna() for c in [1, 2, 3]]
for ax, log_scale in zip(axes, [False, True]):
    ax.boxplot(fare_by_class, tick_labels=['1', '2', '3'])
    if log_scale:
        ax.set_yscale('log')
    ax.set(title=('Цена билета по классу' + (' — log-шкала' if log_scale else '')), xlabel='Класс', ylabel='Цена билета')
plt.tight_layout(); plt.show()

## 2.5. Линия: только когда порядок имеет смысл

Линия соединяет соседние точки, поэтому она подходит для времени, возраста или другого упорядоченного признака. Соединять линией случайные строки DataFrame нельзя: порядок строк не несёт смысла.

Ниже мы сначала создаём возрастные группы через `pd.cut`, затем считаем среднюю `survived` внутри каждой — только после этого рисуем линию. Так каждая точка линии обозначает долю выживших в группе, а не одного пассажира.

In [ ]:
age_data = df.dropna(subset=['age']).copy()
age_data['age_group'] = pd.cut(age_data['age'], bins=[0, 12, 18, 30, 45, 60, 100], right=False)
by_age = age_data.groupby('age_group', observed=True)['survived'].agg(['mean', 'count'])
print(by_age.rename(columns={'mean': 'доля выживших', 'count': 'пассажиров'}))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(by_age.index.astype(str), by_age['mean'], marker='o', color='darkorange', linewidth=2)
ax.set(title='Доля выживших в возрастных группах', xlabel='Возрастная группа', ylabel='Доля', ylim=(0, 1))
ax.grid(alpha=.3); plt.xticks(rotation=20); plt.show()

## Шпаргалка matplotlib

| Вопрос | График | Команда | Важные параметры |
|---|---|---|---|
| Как распределено число? | гистограмма | `ax.hist(x)` | `bins`, `color`, `edgecolor` |
| Сколько объектов в группах? | столбики | `ax.bar(names, counts)` | `color`, `width` |
| Как отличается среднее/доля? | столбики | `ax.bar(names, means)` | `ylim=(0, 1)` для долей |
| Есть ли связь двух чисел? | точки | `ax.scatter(x, y)` | `alpha`, `s`, `c` |
| Как различаются распределения? | boxplot | `ax.boxplot(groups)` | `tick_labels`, `showfliers` |
| Как меняется величина в порядке? | линия | `ax.plot(x, y)` | `marker`, `linewidth` |
| Несколько графиков? | подграфики | `plt.subplots(2, 2)` | `figsize`, `sharex`, `sharey` |

Общие настройки: `ax.set(title=..., xlabel=..., ylabel=...)`, `ax.legend()`, `ax.grid(alpha=.3)`, `ax.set_xlim(...)`, `ax.set_ylim(...)`, `plt.tight_layout()`.

# 3. seaborn: короткие команды для статистических графиков

Seaborn работает поверх matplotlib: он принимает DataFrame прямо целиком, умеет сам группировать столбцы и выглядит аккуратно по умолчанию. Обычно seaborn быстрее для исследования таблицы, а matplotlib удобнее для тонкой ручной настройки. Их можно смешивать: `ax = sns.barplot(...)`, затем `ax.set(...)`.

В seaborn часто встречаются аргументы:

- `data=df` — какая таблица;
- `x='age'`, `y='fare'` — имена столбцов;
- `hue='sex'` — что кодировать цветом;
- `col='class'` — отдельный график для каждой категории;
- `kind='...'` — вид графика в более универсальных функциях.

In [ ]:
# countplot сам считает строки в категориях.
fig, ax = plt.subplots(figsize=(7, 4))
sns.countplot(data=df, x='class', hue='sex', order=['First', 'Second', 'Third'], ax=ax)
ax.set(title='Число пассажиров: класс и пол', xlabel='Класс', ylabel='Число пассажиров')
plt.show()

## 3.1. countplot и barplot — похожи, но отвечают на разные вопросы

- `sns.countplot(...)` **считает строки** в категориях.
- `sns.barplot(..., y='survived')` по умолчанию показывает **среднее** `survived`, то есть долю выживших.

В новых версиях seaborn у `barplot` могут отображаться интервалы неопределённости. Для простого учебного графика можно убрать их параметром `errorbar=None`; тогда читатель видит только среднее.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=df, x='class', y='survived', hue='sex', order=['First', 'Second', 'Third'], errorbar=None, ax=ax)
ax.set(title='Доля выживших: класс и пол', xlabel='Класс', ylabel='Доля', ylim=(0, 1))
ax.legend(title='Пол')
plt.show()

## 3.2. histplot, scatterplot и hue

`hue` добавляет цветную категорию. Он помогает, если категории не слишком многочисленны: два-три цвета читаются хорошо, двадцать — уже нет.

На гистограмме `multiple='layer'` накладывает распределения с прозрачностью; `multiple='stack'` складывает их. На точечном графике `style='sex'` добавляет форму маркера, но использовать одновременно слишком много визуальных кодов стоит только при необходимости.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(data=df, x='age', hue='pclass', bins=20, multiple='layer', alpha=.45, ax=axes[0])
axes[0].set(title='Возраст по классу билета', xlabel='Возраст, лет', ylabel='Число пассажиров')

sns.scatterplot(data=df, x='age', y='fare', hue='survived', style='sex', alpha=.65, ax=axes[1])
axes[1].set(title='Возраст и цена: цвет — исход, форма — пол', xlabel='Возраст, лет', ylabel='Цена билета')
plt.tight_layout(); plt.show()

## 3.3. Тепловая карта: матрица чисел

Тепловая карта (`heatmap`) рисует таблицу чисел цветом. Сначала нужно создать именно числовую таблицу — например, через `pivot_table`. В нашей матрице строки — пол, столбцы — класс билета, ячейка — средняя `survived`.

`annot=True` печатает число в ячейке; `fmt='.2f'` оставляет две цифры после запятой; `vmin=0, vmax=1` делают одну и ту же цветовую шкалу для долей.

In [ ]:
matrix = df.pivot_table(index='sex', columns='class', values='survived', aggfunc='mean', observed=True)
print(matrix)
fig, ax = plt.subplots(figsize=(7, 3))
sns.heatmap(matrix, annot=True, fmt='.2f', vmin=0, vmax=1, cmap='YlGnBu', ax=ax)
ax.set(title='Доля выживших: пол × класс', xlabel='Класс билета', ylabel='Пол')
plt.show()

## Шпаргалка seaborn

| Задача | Команда | Полезные аргументы |
|---|---|---|
| Посчитать категории | `sns.countplot` | `data`, `x`, `hue`, `order` |
| Среднее по группам | `sns.barplot` | `x`, `y`, `hue`, `errorbar=None` |
| Распределение | `sns.histplot` | `x`, `hue`, `bins`, `multiple`, `alpha` |
| Связь чисел | `sns.scatterplot` | `x`, `y`, `hue`, `style`, `size` |
| Категория + число | `sns.boxplot` | `x`, `y`, `hue` |
| Матрица чисел | `sns.heatmap` | `annot`, `fmt`, `cmap`, `vmin`, `vmax` |
| Много похожих графиков | `sns.catplot` / `sns.relplot` | `col`, `row`, `kind`, `hue` |

После любого seaborn-графика остаётся доступен обычный matplotlib-объект `ax`: добавляйте `ax.set(...)`, `ax.legend(...)`, `ax.set_ylim(...)`.

# 4. Как не обмануть себя графиком

1. **Оси.** Подпишите их и не смешивайте количество с долей.
2. **Шкала.** Для долей начинайте с 0; для сильно растянутых чисел честно подпишите log-шкалу.
3. **Размер группы.** Рядом с процентом полезно знать, сколько объектов в группе.
4. **Пропуски.** `dropna()` меняет выборку; сообщайте себе, что часть строк не попала в график.
5. **Причина ≠ связь.** Если в одном классе доля выше, это не доказывает, что класс сам является причиной.
6. **Цвет.** Одинаковый цвет — одинаковый смысл на всех связанных графиках.
7. **Вывод.** Формулируйте его осторожно: «в этой выборке наблюдается…».

# 5. Практика: 25 задач только на визуализацию

Все задания принимают таблицу `df` и **возвращают** графический объект: `Axes`, `Figure` или `FacetGrid`. Тесты не проверяют точный цвет пикселей: они проверяют, что выбран правильный тип графика, данные не испорчены, есть подписи и нужная структура.

Задачи 1–10 закрепляют основу, 11–18 учат читать группы и шкалы, 19–25 собирают исследовательские графики и дашборды. Стратегия: сначала напишите вопрос графика словами; затем создайте `fig, ax`; после рисования добавьте заголовок и подписи; только потом запускайте тест. Если не помните аргумент, он уже встречался в теории выше — интернет не нужен.

In [ ]:
from tests_visualization import *

### Задача 1. Гистограмма возраста — matplotlib

`make_age_hist(df)` должна вернуть `Axes` с гистограммой возраста. Уберите пропуски, используйте не меньше 10 корзинок, добавьте заголовок и подписи X/Y.

Шаблон: `fig, ax = plt.subplots(); ax.hist(...); ax.set(...); return ax`. Проверка: `test_task1(make_age_hist)`.

In [ ]:
def make_age_hist(df):
    # ваш код
    pass

# test_task1(make_age_hist)

### Задача 2. Столбики доли выживших — matplotlib

`make_survival_bar(df)` должна показать долю выживших по полу. Сначала постройте `df.groupby('sex')['survived'].mean()`, затем передайте индексы и значения в `ax.bar`. Установите Y от 0 до 1 и верните `ax`.

Проверка: `test_task2(make_survival_bar)`.

In [ ]:
def make_survival_bar(df):
    # ваш код
    pass

# test_task2(make_survival_bar)

### Задача 3. Точечный график — matplotlib

`make_age_fare_scatter(df)` рисует `age` по X и `fare` по Y. Добавьте `alpha`, заголовок, подписи и верните `ax`. Подумайте: почему здесь не стоит соединять точки линией?

Проверка: `test_task3(make_age_fare_scatter)`.

In [ ]:
def make_age_fare_scatter(df):
    # ваш код
    pass

# test_task3(make_age_fare_scatter)

### Задача 4. Boxplot цены билета — matplotlib

`make_fare_boxplot(df)` строит три распределения `fare` для классов 1, 2 и 3. Используйте список из трёх Series и `ax.boxplot(..., tick_labels=['1', '2', '3'])`. Добавьте заголовок и верните `ax`.

Проверка: `test_task4(make_fare_boxplot)`.

In [ ]:
def make_fare_boxplot(df):
    # ваш код
    pass

# test_task4(make_fare_boxplot)

### Задача 5. Среднее по категориям — seaborn

`seaborn_class_bar(df)` строит `sns.barplot`: X — `pclass`, Y — `survived`, `estimator='mean'` можно не писать, потому что это значение по умолчанию. Уберите интервалы `errorbar=None`, задайте Y от 0 до 1, подпишите оси и верните `ax`.

Проверка: `test_task5(seaborn_class_bar)`.

In [ ]:
def seaborn_class_bar(df):
    # ваш код
    pass

# test_task5(seaborn_class_bar)

### Задача 6. Цвет как переменная — seaborn

`seaborn_survival_scatter(df)` рисует seaborn scatter: X — возраст, Y — цена, `hue='survived'`. Нужна легенда, поэтому не отключайте её. Подпишите график и верните `ax`.

Проверка: `test_task6(seaborn_survival_scatter)`.

In [ ]:
def seaborn_survival_scatter(df):
    # ваш код
    pass

# test_task6(seaborn_survival_scatter)

### Задача 7. Несколько распределений — seaborn

`seaborn_age_hist(df)` строит гистограмму возраста с `hue='pclass'`. Используйте `sns.histplot`, не забудьте про `bins` и верните `ax`. Посмотрите, читается ли легенда.

Проверка: `test_task7(seaborn_age_hist)`.

In [ ]:
def seaborn_age_hist(df):
    # ваш код
    pass

# test_task7(seaborn_age_hist)

### Задача 8. Линия после агрегации

`plot_age_group_rate(df)` создаёт возрастные группы любым разумным способом (`pd.cut`), считает среднюю `survived` для каждой и рисует линию с маркерами. Y — от 0 до 1. Верните `ax`.

Проверка: `test_task8(plot_age_group_rate)`.

In [ ]:
def plot_age_group_rate(df):
    # ваш код
    pass

# test_task8(plot_age_group_rate)

### Задача 9. Тепловая карта — seaborn

`seaborn_survival_heatmap(df)` строит матрицу средней `survived` для `sex × pclass`, используя `pivot_table`, затем `sns.heatmap`. Добавьте `annot=True`, цветовую шкалу 0–1, подписи осей и верните `ax`.

Проверка: `test_task9(seaborn_survival_heatmap)`.

In [ ]:
def seaborn_survival_heatmap(df):
    # ваш код
    pass

# test_task9(seaborn_survival_heatmap)

### Задача 10. Мини-дашборд

`make_dashboard(df)` возвращает `Figure` с сеткой 2×2. Обязательные четыре графика: гистограмма возраста, столбики доли выживших по классу, scatter `age/fare`, heatmap `sex × pclass`. У каждого графика должен быть заголовок.

Подсказка: `fig, axes = plt.subplots(2, 2, figsize=(...))`, затем обращайтесь к `axes[0, 0]`, `axes[0, 1]` и так далее. Проверка: `test_task10(make_dashboard)`.

In [ ]:
def make_dashboard(df):
    # ваш код
    pass

# test_task10(make_dashboard)

## Часть B. Ещё 15 задач: от уверенного владения к мини-исследованию

В этих заданиях уже меньше пошаговых подсказок. Перед каждым графиком формулируйте для себя: что означает один объект, что означают оси и что будет означать цвет. После построения обязательно запускайте соответствующий тест.

### Задача 11. Countplot порта посадки

Функция count_by_embarked(df) строит sns.countplot для embarked. Укажите осмысленный порядок портов C, Q, S, заголовок и подписи. Это график количества, не доли. Верните ax.

Проверка: test_task11(count_by_embarked).

In [ ]:
def count_by_embarked(df):
    # ваш код
    pass

# test_task11(count_by_embarked)

### Задача 12. Scatter с log-шкалой

Функция fare_log_scatter(df) рисует точки age/fare, но цена по Y должна быть в логарифмической шкале. Это помогает увидеть и дешёвые билеты, и дорогие выбросы одновременно. Используйте ax.set_yscale('log'), подпишите шкалу и верните ax.

Проверка: test_task12(fare_log_scatter).

In [ ]:
def fare_log_scatter(df):
    # ваш код
    pass

# test_task12(fare_log_scatter)

### Задача 13. Boxplot возраста — seaborn

Функция seaborn_age_boxplot(df) сравнивает возраст в трёх классах билета через sns.boxplot. По X должен быть pclass, по Y age. Добавьте заголовок и верните ax. Обратите внимание: seaborn сам пропускает NaN на оси Y.

Проверка: test_task13(seaborn_age_boxplot).

In [ ]:
def seaborn_age_boxplot(df):
    # ваш код
    pass

# test_task13(seaborn_age_boxplot)

### Задача 14. Групповые столбики

Функция seaborn_survival_grouped_bar(df) строит barplot: X — pclass, Y — survived, цвет — sex. Уберите errorbar, задайте Y от 0 до 1, добавьте легенду и верните ax. Этот график одновременно сравнивает класс и пол.

Проверка: test_task14(seaborn_survival_grouped_bar).

In [ ]:
def seaborn_survival_grouped_bar(df):
    # ваш код
    pass

# test_task14(seaborn_survival_grouped_bar)

### Задача 15. Подписать важную точку

Функция annotate_max_fare(df) строит scatter возраст/цена и стрелкой или текстом подписывает пассажира с максимальным fare. Найдите строку через idxmax или sort_values. Верните ax.

Проверка: test_task15(annotate_max_fare).

In [ ]:
def annotate_max_fare(df):
    # ваш код
    pass

# test_task15(annotate_max_fare)

### Задача 16. Две линии после агрегации

Функция survival_lines_by_sex(df) создаёт возрастные группы и рисует две линии доли выживших: отдельно для female и male. Сначала сгруппируйте данные по возрастной группе и полу, затем нарисуйте линии с label и легендой. Y от 0 до 1, верните ax.

Проверка: test_task16(survival_lines_by_sex).

In [ ]:
def survival_lines_by_sex(df):
    # ваш код
    pass

# test_task16(survival_lines_by_sex)

### Задача 17. Кривая плотности KDE

Функция seaborn_age_kde(df) использует sns.kdeplot для возраста и разделяет линии по sex через hue. Добавьте заголовок, подписи и верните ax. KDE — сглаженная оценка распределения, поэтому не воспринимайте её как точное число пассажиров.

Проверка: test_task17(seaborn_age_kde).

In [ ]:
def seaborn_age_kde(df):
    # ваш код
    pass

# test_task17(seaborn_age_kde)

### Задача 18. Heatmap: порт и класс

Функция embarked_class_heatmap(df) создаёт pivot_table: строки embarked, столбцы pclass, значения survived и aggfunc='mean'. Постройте heatmap с annot=True и шкалой 0–1, подпишите оси, верните ax.

Проверка: test_task18(embarked_class_heatmap).

In [ ]:
def embarked_class_heatmap(df):
    # ваш код
    pass

# test_task18(embarked_class_heatmap)

### Задача 19. Малые множества графиков (facets)

Функция facet_age_by_class(df) возвращает объект seaborn FacetGrid минимум с двумя отдельными графиками возраста по классу. Удобный путь: sns.displot(data=df, x='age', col='pclass', bins=...). Не вызывайте plt.show внутри функции.

Проверка: test_task19(facet_age_by_class).

In [ ]:
def facet_age_by_class(df):
    # ваш код
    pass

# test_task19(facet_age_by_class)

### Задача 20. Матрица корреляций

Функция correlation_heatmap(df) выбирает только числовые столбцы, считает .corr() и показывает heatmap. Используйте cmap с двумя направлениями, например coolwarm, center=0, добавьте заголовок и верните ax. Корреляция показывает линейную связь чисел, но не причину.

Проверка: test_task20(correlation_heatmap).

In [ ]:
def correlation_heatmap(df):
    # ваш код
    pass

# test_task20(correlation_heatmap)

### Задача 21. Круговая диаграмма — только для двух долей

Функция survival_pie(df) строит pie с двумя секторами: не выжил и выжил. Возьмите df['survived'].value_counts().sort_index(), подпишите сектора через labels и проценты через autopct. Добавьте заголовок, верните ax. Это упражнение полезно для практики, но в отчётах barplot часто читается лучше.

Проверка: test_task21(survival_pie).

In [ ]:
def survival_pie(df):
    # ваш код
    pass

# test_task21(survival_pie)

### Задача 22. Семейные группы и линия

Функция survival_by_sibsp(df) группирует по sibsp, считает среднюю survived и число пассажиров, рисует линию с маркерами. Добавьте оси, Y от 0 до 1 и верните ax. Отдельно посмотрите на число пассажиров: крайние точки могут быть ненадёжны из-за маленькой группы.

Проверка: test_task22(survival_by_sibsp).

In [ ]:
def survival_by_sibsp(df):
    # ваш код
    pass

# test_task22(survival_by_sibsp)

### Задача 23. Violinplot цены билета

Функция seaborn_fare_violin(df) строит sns.violinplot: X — pclass, Y — fare. Добавьте заголовок и верните ax. Violinplot похож на boxplot, но ширина фигуры показывает плотность значений. При длинном хвосте можно добавить log-шкалу, но это не обязательно.

Проверка: test_task23(seaborn_fare_violin).

In [ ]:
def seaborn_fare_violin(df):
    # ваш код
    pass

# test_task23(seaborn_fare_violin)

### Задача 24. Собственный стиль

Функция styled_class_count(df) строит countplot классов билета с собственной палитрой из трёх цветов. Используйте palette со списком цветов, добавьте заголовок и верните ax. Главное правило: цвет не должен мешать чтению и должен иметь понятный смысл.

Проверка: test_task24(styled_class_count).

In [ ]:
def styled_class_count(df):
    # ваш код
    pass

# test_task24(styled_class_count)

### Задача 25. Финальный исследовательский дашборд

Функция make_final_dashboard(df) возвращает Figure с сеткой 2×3 и шестью графиками. Включите: распределение возраста, countplot класса, долю выживших по полу, scatter возраст/цена, boxplot цены по классу и heatmap sex×pclass. У каждого графика должен быть заголовок; добавьте общий заголовок fig.suptitle и вызовите fig.tight_layout. Это итоговая работа: постарайтесь, чтобы дашборд можно было показать человеку без контекста.

Проверка: test_task25(make_final_dashboard).

In [ ]:
def make_final_dashboard(df):
    # ваш код
    pass

# test_task25(make_final_dashboard)

# Итоговая памятка

**Выбираю график:** одно число → hist; категория и количество → count/bar; категория и среднее/доля → barplot; два числа → scatter; несколько распределений → boxplot/histplot; таблица чисел → heatmap.

**Перед сдачей:** у каждого графика есть вопрос, заголовок и подписи; на долях Y идёт от 0 до 1; пропуски не превращены в нули; одинаковые цвета имеют одинаковый смысл; вывод не путает связь и причину.

Следующий урок — [машинное обучение](../class09:%20machine%20learning/machine_learning_basics.ipynb): там визуализация пригодится, чтобы понять данные до обучения модели.